# Papalexi ECCITE-seq demo — preprocessing/QC with petrubseq-pipeline-protein

**Documentation only.** This notebook does not process anything; the processing is done by the production CLI (`petrubseq-protein run`). It describes the dataset, shows the demo configuration and command, loads the processed object written by the CLI and displays a few of its QC tables. The full report is `report.html`.

## 1. Dataset

Papalexi E. *et al.* (2021) *Characterizing the molecular regulation of inhibitory immune checkpoints with multimodal single-cell screens*, Nat Genet 53:322–331 — GEO **GSE153056**. The demo uses the **pooled ECCITE-seq screen** (GSM4633614–GSM4633618): IFN-γ stimulated THP-1 cells carrying one sgRNA each from a 111-guide library (25 gene targets × 3–4 guides, 9 non-targeting `NT` guides, one `eGFP` guide), profiled for RNA, 4 surface proteins (CD86, PD-L1, PD-L2, CD366) and the guide itself (GDO). The audit of the local files is in `docs/datasets/PAPALEXI_ECCITE_AUDIT.md`.

`demo/prepare_papalexi_demo.py` writes a deterministic stratified subset (seed 0; 1,800 HTO-singlet cells over (HTO sample × target) strata; raw counts unchanged) as a 10x-style combined matrix plus per-cell metadata, an antibody table and a guide→target table under `${DATA_ROOT}/demo/papalexi_eccite/`. What the demo does **not** do: HTO demultiplexing beyond a simple singlet rule, no reproduction of the paper's mixscape / perturbation analyses, no stimulated-vs-control contrast (every cell of the pooled screen is IFN-γ stimulated).

## 2. Configuration and production command

The demo runs through the generic Stage C architecture: one `inputs.multiplexed` MTX matrix split by feature type; guides called from the guide counts (dominant rule); raw ADT counts → CLR.

In [ ]:
from pathlib import Path
REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
print((REPO / "config" / "demo_papalexi.yaml").read_text())

```bash
conda activate petrubseq-protein
python demo/prepare_papalexi_demo.py --source ${DATA_ROOT}/ECCITE-seq --output ${DATA_ROOT}/demo/papalexi_eccite   # once
petrubseq-protein run --config config/demo_papalexi.yaml                                                          # ~90 s, < 1 GB
python scripts/validate_processed.py ${RESULTS_ROOT}/demo/papalexi_eccite/processed/papalexi_eccite_demo_processed.h5ad
```
On this HPC both steps run through `slurm/prepare_papalexi_demo.slurm` and `slurm/run_demo_papalexi.slurm` (see `docs/HPC.md`).

## 3. The processed object

In [ ]:
import os
import anndata as ad, pandas as pd
RESULTS = Path(os.environ.get("PETRUBSEQ_RESULTS_ROOT", REPO.parent / "results")) / "demo" / "papalexi_eccite"
adata = ad.read_h5ad(RESULTS / "processed" / "papalexi_eccite_demo_processed.h5ad")
adata

In [ ]:
u = adata.uns["petrubseq_protein"]
pd.DataFrame({"input_state": [u["rna"]["input_state_used"]], "rna_method": [u["rna"]["method"]], "protein_primary": [u["protein"]["primary_method"]],
              "assignment_source": [u["perturbations"]["assignment"]["effective_source"]], "guide_rule": [u["perturbations"]["assignment"]["guide_calling"]["method"]],
              "cells": [adata.n_obs], "genes": [adata.n_vars], "guides": [adata.obsm["guide_counts"].shape[1]]}).T

## 4. Selected QC outputs (written by the CLI under `tables/`)

In [ ]:
pd.read_csv(RESULTS / "tables" / "qc_filtering_steps.csv")

In [ ]:
adata.obs["perturbation_class"].value_counts().to_frame("cells")

In [ ]:
pd.read_csv(RESULTS / "tables" / "target_coverage.csv").sort_values("n_cells_single_guide", ascending=False).head(12)

In [ ]:
adata.uns["protein_features"][["role", "antibody_name", "gene_symbol", "annotation_source", "used_in_embedding"]]

In [ ]:
pd.read_csv(RESULTS / "tables" / "perturbation_qc.csv").head(20)

## 5. Report

Every figure (RNA/protein QC before and after filtering, perturbation and guide-count diagnostics, representations, cross-modality diagnostics) is embedded in `report.html` next to the processed object; `report.md` mirrors it.

In [ ]:
print(RESULTS / "report.html", (RESULTS / "report.html").exists())
pd.read_csv(RESULTS / "tables" / "figure_manifest.csv")[["section", "stage", "name"]].groupby(["section", "stage"]).size().to_frame("figures")